# Fine-tuning sur Colab — captures réelles (pipeline retrain.py)

Reproduit **exactement** le pipeline local, sur GPU Colab (T4) : split par disposition
(test compost jamais appris) → éval **avant** → fine-tuning → éval **après**, via
`scripts/retrain.py` — aucune logique dupliquée dans ce notebook.

**Prérequis (une seule fois)** :
- secret `GITHUB_TOKEN` dans Colab (icône clé à gauche) ;
- le zip du snapshot sur Drive : `MyDrive/compost/dataset_captures_v003.zip` ;
- GPU activé : Exécution → Modifier le type d'exécution → T4.

Durée indicative : **~1 h sur v002 (440 img), compter ~2-3 h sur v003 (1031 img)**
(RT-DETR batch 4). Le run est sauvegardé sur Drive **toutes les 5 epochs** ; si Colab
coupe, la commande de REPRISE est en commentaire de la cellule 5 (perte max ~5 epochs).


In [ ]:
# 1. Clone du repo
BRANCH = 'docker'   # branche de travail ; mettre 'main' après fusion
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
%cd /content
!rm -rf /content/repo
!git clone --depth 1 --branch {BRANCH} https://{token}@github.com/TSResearch-hub/Compost_Waste_Yolo.git /content/repo
%cd /content/repo


In [ ]:
# 2. Installation des dépendances (+ police des planches d'audit, absente de Colab)
!pip install -q -e .
!apt-get -qq install -y fonts-dejavu-core > /dev/null


In [ ]:
# 3. Paramètres + montage Drive
# Pré-entraîné canonique = v0 (models/v0_pretrain_rtdetr-l.pt en local, voir
# models/README.md). Sur Drive il vit sous le nom de son run d'origine :
PRETRAIN_PATH = '/content/drive/MyDrive/compost/backups/pretrain_16-07_03h44/weights/best.pt'
# Zip du snapshot de captures (copie de data/captures/vNNN du PC, voir README) :
SNAPSHOT_ZIP  = '/content/drive/MyDrive/compost/dataset_captures_v003.zip'
EPOCHS = 30   # ~25-30 suffisaient sur v002 (~440 img) ; à surveiller sur v003 (~1030)
BATCH  = 4    # RT-DETR s'entraîne à 1280 px (mosaïque) : 4 tient sur T4 ; YOLO : 16

import os, shutil
from google.colab import drive
if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')
assert os.path.exists(PRETRAIN_PATH), f'Pré-entraîné introuvable : {PRETRAIN_PATH}'
assert os.path.exists(SNAPSHOT_ZIP),  f'Zip du snapshot introuvable : {SNAPSHOT_ZIP}'


In [ ]:
# 4. Décompression du snapshot + contrôle (mêmes comptes que le snapshot local)
!rm -rf /content/captures_snapshot && mkdir -p /content/captures_snapshot
!unzip -q {SNAPSHOT_ZIP} -d /content/captures_snapshot
from pathlib import Path
snap = Path('/content/captures_snapshot')
if not (snap / 'images').is_dir():   # zip fait depuis le dossier parent du snapshot
    snap = next(d for d in snap.iterdir() if (d / 'images').is_dir())
n_img = sum(1 for f in (snap / 'images').iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
n_lbl = len(list((snap / 'labels').glob('*.txt')))
print(f'{n_img} images, {n_lbl} labels — attendu : 1031 / 741 (snapshot v003_21-07)')
assert (snap / 'groups.csv').exists(), 'groups.csv manquant : split par disposition impossible'
assert (n_img, n_lbl) == (1031, 741), 'Comptes inattendus — mauvais zip ? (adapter si nouveau snapshot)'
SNAP = str(snap)


In [ ]:
# 5. Pipeline complet — identique au local (split -> éval avant -> fine-tuning -> éval après).
#    La comparaison avant/après s'affiche à la fin de la sortie.
#    Backup du run vers Drive toutes les 5 epochs (perte max ~5 epochs si Colab coupe).
BACKUP_DIR = '/content/drive/MyDrive/compost/backups'
!python scripts/retrain.py --pretrain {PRETRAIN_PATH} --captures {SNAP} \
    --epochs {EPOCHS} --batch {BATCH} --runs-dir /content/runs \
    --backup-dir {BACKUP_DIR} --backup-every 5

# Après coupure Colab : relancer les cellules 1-4, puis SEULEMENT les commandes
# ci-dessous (PAS retrain.py : il repartirait de zéro). Le split est déterministe
# (même seed) donc on le recrée à l'identique, puis train.py REPREND depuis le
# last.pt sauvegardé sur Drive (remplacer <run> par le nom, ex. finetune_21-07_10h12) :
#   !python scripts/split_captures.py --source {SNAP} --output data/finetune --seed 42
#   !python scripts/prepare_dataset.py --source data/finetune/captures_finetune \
#       --output data/finetune/dataset_finetune --ratios 0.85 0.15 0 --symlink
#   !python scripts/train.py --resume {BACKUP_DIR}/finetune_<run>/weights/last.pt \
#       --backup-dir {BACKUP_DIR} --backup-every 5
# puis l'éval APRÈS (la comparaison avant/après de retrain.py n'aura pas tourné) :
#   !python scripts/evaluate.py --weights /content/runs/finetune_<run>/weights/best.pt \
#       --data data/finetune/captures_test/data.yaml --split test --runs-dir /content/runs


In [ ]:
# 6. Figures des évaluations (avant = eval_pretrain_*, après = eval_finetune_*)
from pathlib import Path
from IPython.display import Image, display
for d in sorted(Path('/content/runs').glob('eval_*')):
    print('\n===', d.name)
    for png in ('per_class_metrics.png', 'confusion_matrices.png'):
        if (d / png).exists():
            display(Image(str(d / png), width=700))


In [ ]:
# 7. Sauvegarde des runs sur Drive + téléchargement direct du best.pt
!mkdir -p /content/drive/MyDrive/compost/runs
!cp -r /content/runs/* /content/drive/MyDrive/compost/runs/
from pathlib import Path
best = max(Path('/content/runs').glob('finetune_*/weights/best.pt'),
           key=lambda p: p.stat().st_mtime)
print('best.pt :', best)
from google.colab import files
files.download(str(best))   # arrive dans les téléchargements du navigateur (~252 Mo)


## Rapatrier et versionner sur le PC

Le `best.pt` téléchargé est un **candidat** de nouvelle version — voir la
convention dans `models/README.md` :

1. Le déposer dans `models/` sous son nom de version
   (ex. `v3_finetune_rtdetr-l_snapshot-v004.pt`), en l'allégeant au passage
   (le checkpoint Colab garde l'optimiseur : 263 Mo au lieu de 66, mêmes poids) :

       python -c "from ultralytics.utils.torch_utils import strip_optimizer; \
           strip_optimizer('best.pt', 'models/v3_finetune_rtdetr-l_snapshot-v004.pt')"

2. Ajouter sa ligne au tableau de `models/README.md` (données, run source, éval).
3. **Seulement s'il est retenu après évaluation**, l'installer comme modèle déployé
   des interfaces :

       cp models/v3_finetune_rtdetr-l_snapshot-v004.pt weights/best.pt

Les runs complets (courbes, évals) sont aussi sur Drive dans `compost/runs/` —
à rapatrier dans `runs/` local pour l'onglet Résultats.
